# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [2]:
import pandas as pd

# Load the dataset required for this lab
url = ("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv")

marketing_data = pd.read_csv(url)

In [3]:
# Standardize column names
marketing_data.columns = (
    marketing_data.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

# Inspect the dataset
print(f"Dataset shape: {marketing_data.shape}")
print(f"Columns: {marketing_data.columns.tolist()}")

display(marketing_data.head())

Dataset shape: (10910, 26)
Columns: ['unnamed:_0', 'customer', 'state', 'customer_lifetime_value', 'response', 'coverage', 'education', 'effective_to_date', 'employmentstatus', 'gender', 'income', 'location_code', 'marital_status', 'monthly_premium_auto', 'months_since_last_claim', 'months_since_policy_inception', 'number_of_open_complaints', 'number_of_policies', 'policy_type', 'policy', 'renew_offer_type', 'sales_channel', 'total_claim_amount', 'vehicle_class', 'vehicle_size', 'vehicle_type']


,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,...,number_of_open_complaints,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2/18/11,Employed,M,...,0.0,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,NaN
1,1,KX64629,California,2228.525238,No,Basic,College,1/18/11,Unemployed,F,...,0.0,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,NaN
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2/10/11,Employed,M,...,0.0,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
4,4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,1/17/11,Medical Leave,F,...,NaN,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,NaN


In [6]:
# Identify customers who responded Yes
responded_yes = (
    marketing_data["response"]
    .str.strip()
    .str.lower()
    .eq("yes")
    .fillna(False)
)

# Select customers meeting both conditions
low_claim_responders = marketing_data.loc[
    (marketing_data["total_claim_amount"] < 1000)
    & responded_yes
].copy()

# Inspect the filtered DataFrame
print(f"Selected records: {len(low_claim_responders)}")

display(
    low_claim_responders[
        ["customer", "response", "total_claim_amount"]
    ].head(10)
)

Selected records: 1399


,customer,response,total_claim_amount
3,XL78013,Yes,484.013411
8,FM55990,Yes,739.200000
15,CW49887,Yes,547.200000
19,NJ54277,Yes,19.575683
27,MQ68407,Yes,60.036683
65,LO57874,Yes,302.400000
67,KR35099,Yes,231.201886
69,QG27547,Yes,374.400000
88,CJ51511,Yes,571.200000
102,VG56765,Yes,114.273025


The filtered DataFrame contains 1,399 records with a total claim amount below 1,000 and a positive response to the latest marketing campaign.

In [7]:
# Select all positive campaign responses from the original DataFrame
positive_responders = marketing_data.loc[responded_yes].copy()

# Compare premiums, CLV, and claims by policy type and gender
segment_summary = (
    positive_responders
    .groupby(["policy_type", "gender"], dropna=False)
    .agg(
        records=("customer", "size"),
        avg_monthly_premium=("monthly_premium_auto", "mean"),
        avg_clv=("customer_lifetime_value", "mean"),
        avg_claim_amount=("total_claim_amount", "mean")
    )
    .round(2)
)

# Display the results without requiring additional libraries
with pd.option_context("display.float_format", "{:,.2f}".format):
    display(segment_summary)

records  avg_monthly_premium  avg_clv  avg_claim_amount
policy_type    gender                                                         
Corporate Auto F           169                94.30 7,712.63            433.74
               M           154                92.19 7,944.47            408.58
Personal Auto  F           540                99.00 8,339.79            452.97
               M           536                91.09 7,448.38            457.01
Special Auto   F            35                92.31 7,691.58            453.28
               M            32                86.34 8,247.09            429.53

### Comparison of customers who responded Yes

-- Female Personal Auto customers have the highest average monthly premium (99.00) and average CLV (8,339.79). Their average claim amount is 452.97, slightly below the male Personal Auto group at 457.01. This segment appears promising in terms of premium income and customer value.

-- Male Corporate Auto customers have the lowest average claim amount (408.58), alongside an average monthly premium of 92.19 and an average CLV of 7,944.47. They appear favorable when considering average claim costs.

-- Male Special Auto customers have a relatively high average CLV (8,247.09), but the lowest average monthly premium (86.34). Both Special Auto groups are small, with 32 male and 35 female records, so their averages should be interpreted cautiously.

These comparisons identify potentially attractive segments, but they do not establish profitability or risk conclusively. Premiums and claims require comparable time periods, and other costs and exposure levels are not included.

In [8]:
# Count distinct customer IDs within each state
customers_by_state = (
    marketing_data
    .groupby("state")["customer"]
    .nunique()
    .rename("customer_count")
    .sort_values(ascending=False)
    .reset_index()
)

print("Customer counts by state:")
display(customers_by_state)

# Keep only states with more than 500 customers
states_over_500 = customers_by_state.loc[
    customers_by_state["customer_count"] > 500
].reset_index(drop=True)

print("\nStates with more than 500 customers:")
display(states_over_500)

Customer counts by state:


,state,customer_count
0,California,3150
1,Oregon,2601
2,Arizona,1703
3,Nevada,882
4,Washington,798



States with more than 500 customers:


,state,customer_count
0,California,3150
1,Oregon,2601
2,Arizona,1703
3,Nevada,882
4,Washington,798


### Customer counts by state

-- All five states exceed the threshold of 500 distinct customers. California has the highest count (3,150), followed by Oregon (2,601), Arizona (1,703), Nevada (882), and Washington (798).

Therefore, filtering for states with more than 500 customers retains all five states.

In [9]:
# Calculate CLV statistics by education level and gender
clv_by_education_gender = (
    marketing_data
    .groupby(["education", "gender"], dropna=False)
    .agg(
        maximum_clv=("customer_lifetime_value", "max"),
        minimum_clv=("customer_lifetime_value", "min"),
        median_clv=("customer_lifetime_value", "median")
    )
)

# Display the statistics with two decimal places
with pd.option_context("display.float_format", "{:,.2f}".format):
    display(clv_by_education_gender)

maximum_clv  minimum_clv  median_clv
education            gender                                      
Bachelor             F         73,225.96     1,904.00    5,640.51
                     M         67,907.27     1,898.01    5,548.03
College              F         61,850.19     1,898.68    5,623.61
                     M         61,134.68     1,918.12    6,005.85
Doctor               F         44,856.11     2,395.57    5,332.46
                     M         32,677.34     2,267.60    5,577.67
High School or Below F         55,277.45     2,144.92    6,039.55
                     M         83,325.38     1,940.98    6,286.73
Master               F         51,016.07     2,417.78    5,729.86
                     M         50,568.26     2,272.31    5,579.10

### Customer lifetime value by education and gender

-- Male customers with High School or Below have the highest median CLV (6,286.73) and the highest maximum CLV (83,325.38). Female customers in the same education group also have the highest median among female groups, at 6,039.55.

-- Female customers with a doctorate have the lowest median CLV among all groups, at 5,332.46. The lowest individual CLV is observed among male Bachelor customers, at 1,898.01.

-- Gender differences depend on educational level. Female customers have higher median CLV in the Bachelor and Master groups, while male customers have higher medians in the College, Doctor, and High School or Below groups.

-- Maximum values are substantially higher than medians across all groups, indicating some customers have exceptionally high CLV relative to their group's typical value.

Overall, the results do not show a consistent increase in median CLV with higher educational attainment. These are descriptive associations and do not establish causation.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [10]:
# Parse the effective date and extract the year-month
effective_dates = pd.to_datetime(
    marketing_data["effective_to_date"],
    format="%m/%d/%y",
    errors="raise"
)

marketing_data["month"] = effective_dates.dt.to_period("M")

# Report records missing a grouping variable
missing_group_values = marketing_data[["state", "month"]].isna().any(axis=1)
print(f"Records excluded due to missing state or month: {missing_group_values.sum()}")

# Count policy records by state and month
policies_by_state_month = (
    marketing_data
    .groupby(["state", "month"])
    .size()
    .unstack("month", fill_value=0)
    .sort_index()
)

display(policies_by_state_month)

Records excluded due to missing state or month: 631


month,2011-01,2011-02
state,,
Arizona,1008,929
California,1918,1634
Nevada,551,442
Oregon,1565,1344
Washington,463,425


In [11]:
# Identify the top three states across all months
total_policies_by_state = policies_by_state_month.sum(axis=1)

top_three_states = (
    total_policies_by_state
    .sort_values(ascending=False)
    .head(3)
    .index
)

# Create a new DataFrame with their monthly policy counts
top_three_states_monthly = (
    policies_by_state_month.loc[top_three_states].copy()
)

display(top_three_states_monthly)

month,2011-01,2011-02
state,,
California,1918,1634
Oregon,1565,1344
Arizona,1008,929


### Policy records by state and month

-- alifornia has the highest total (3,552), followed by Oregon (2,909) and Arizona (1,937). These three states were selected for the monthly comparison.

-- All five states have fewer policy records in February than in January. These are monthly totals rather than daily sales rates.

-- The analysis excludes 631 records with a missing state or month. Each record is treated as one policy for this exercise.

In [12]:
# Select records with a known channel and response
channel_data = marketing_data[
    ["sales_channel", "response"]
].dropna().copy()

# Standardize response labels
channel_data["response"] = (
    channel_data["response"].str.strip().str.title()
)

channel_data = channel_data.loc[
    channel_data["response"].isin(["Yes", "No"])
].copy()

print(f"Records excluded: {len(marketing_data) - len(channel_data)}")

# Count responses within each channel
response_counts = pd.crosstab(
    channel_data["sales_channel"],
    channel_data["response"]
).reindex(columns=["No", "Yes"], fill_value=0)

# Calculate response percentages within each channel
response_rates = response_counts.div(
    response_counts.sum(axis=1),
    axis=0
).mul(100)

# Convert the percentage table from wide to long format
response_rates_long = response_rates.reset_index().melt(
    id_vars="sales_channel",
    var_name="response",
    value_name="response_rate_percent"
)

# Select and rank positive response rates
yes_response_rates = (
    response_rates_long.loc[
        response_rates_long["response"] == "Yes"
    ]
    .sort_values("response_rate_percent", ascending=False)
    .reset_index(drop=True)
)

with pd.option_context("display.float_format", "{:,.2f}".format):
    display(yes_response_rates)

Records excluded: 631


,sales_channel,response,response_rate_percent
0,Agent,Yes,19.07
1,Web,Yes,11.71
2,Branch,Yes,11.38
3,Call Center,Yes,10.98


### Positive response rates by sales channel

-- Agent has the highest positive response rate at 19.07%, followed by Web at 11.71%, Branch at 11.38%, and Call Center at 10.98%.

-- Agent exceeds Call Center by 8.09 percentage points. Web, Branch, and Call Center have relatively similar response rates.

-- The calculation excludes 631 records without a usable channel or response. Percentages are calculated within each channel using known Yes and No responses.

-- These results show an association between sales channel and campaign response, but they do not establish that the channel caused the difference.